Copyright 2026 Google LLC

SPDX-License-Identifier: Apache-2.0

주의: 본 코드는 상용 배포용이 아닌 학습 및 데모용 가이드다.

용도: GCP 상에서 최근 발생한 권한 거부(Permission Denied ) 실패 에러 감사 로그를 실시간으로 역추적하여, 해당 실패를 즉시 해결하기 위해 필요한 최소한의 구체적인 IAM 역할을 진단 및 처방하는 지능형 해결사 도구다.

## IAM 권한 실패 해결사 (IAM Permission Resolver )

### 1. 의존성 패키지 설치

GCP 로깅 감사 로그를 정밀 분석하기 위해 필요한 핵심 클라우드 라이브러리를 설치한다.

In [ ]:
!pip install --quiet google-cloud-logging

### 2. 활성 GCP 프로젝트 ID 동적 탐색

현재 활성화되어 있는 사용자의 자격 증명을 기반으로 GCP 프로젝트 ID를 동적으로 검색한다. 자격 증명이 유효하지 않은 경우 수동으로 구성할 수 있다. (GCP 콘솔 자격증명 참조 주소: https://console.cloud.google.com/ )


In [ ]:
import google.auth

try:
  _, project_id = google.auth.default()
  if not project_id:
    raise ValueError("프로젝트 ID를 탐색하지 못했다.")
  print(f"[성공] 활성화된 GCP 프로젝트 ID 감지: {project_id}")
except Exception as e:
  print("[경고] 자격 증명을 통해 프로젝트 ID를 찾지 못했다. 수동으로 설정해야 한다. (GCP 콘솔 자격증명 참조 주소: https://console.cloud.google.com/ )")
  project_id = "your-project-id" # 본인의 실제 GCP 프로젝트 ID로 변경하기 바란다.


### 3. 권한 실패 감사 로그 수집 및 정밀 진단

GCP 감사 로그로부터 데이터 액세스 권한 거부 실패 내역을 역추적하여 권장하는 IAM 역할을 도출하고 즉시 대응 가능한 gcloud 해결 명령어를 출력한다.

**상세 분석 흐름 및 규칙:**
* **감사 로그 필터링**: 감사 로그 수집 시 에러 코드가 7(Permission Denied )이거나 상태 메시지에 권한 거부 관련 텍스트(`Permission denied`, `Forbidden` )가 포함된 데이터 액세스 로그만 수집한다.
* **서비스별 권장 IAM 역할 매핑**: 수집된 서비스의 명칭(예: aiplatform, bigquery, storage, logging )에 알맞은 최적의 권장 역할과 역할 정보를 정의하여 출력한다.
* **gcloud 복구 명령 조립**: 계정 유형(사용자 또는 서비스 계정 )을 자동으로 식별하여 대상 프로젝트에 바인딩할 수 있는 add-iam-policy-binding 쉘 구문을 생성한다.

In [ ]:
from datetime import datetime, timedelta, timezone
from google.cloud import logging_v2

DAYS = 7
LIMIT_COUNT = 5

client = logging_v2.Client(project=project_id)
start_date = (datetime.now(timezone.utc) - timedelta(days=DAYS)).strftime("%Y-%m-%dT%H:%M:%SZ")

log_filter = (
  f'logName="projects/{project_id}/logs/cloudaudit.googleapis.com%2Fdata_access" '
  f'AND (protoPayload.status.code=7 OR protoPayload.status.message:"Permission denied" OR protoPayload.status.message:"Forbidden") '
  f'AND timestamp>="{start_date}"'
)

print(f"[자가 진단 시작] 최근 발생한 GCP 권한 실패 감사 로그 역추적을 시작한다...")
print(f"조회 대상 기간: 최근 {DAYS}일")
print(f"조회 대상 로그 개수: 최대 {LIMIT_COUNT}개\n" + "-"*72)

try:
  entries = client.list_entries(filter_=log_filter, order_by="timestamp desc", page_size=LIMIT_COUNT)
  found_attempts = 0
  
  for entry in entries:
    if found_attempts >= LIMIT_COUNT:
      break
      
    payload = entry.proto_payload
    if not payload:
      continue
      
    principal_email = payload.get("authenticationInfo", {}).get("principalEmail")
    service_name = payload.get("serviceName")
    method_name = payload.get("methodName")
    status_message = payload.get("status", {}).get("message", "상세 사유 누락")
    
    if not principal_email or not service_name:
      continue
      
    found_attempts += 1
    print(f"[실패 건 #{found_attempts}]")
    print(f"  - 실패 주체 계정: {principal_email}")
    print(f"  - 요청 대상 서비스: {service_name}")
    print(f"  - 실행 실패 액션: {method_name}")
    print(f"  - 실제 오류 내용: {status_message}\n")
    
    recommended_role = "roles/viewer"
    role_desc = "서비스 조회에 필요한 기본 프로젝트 뷰어 권한이다."
    
    if service_name == "aiplatform.googleapis.com":
      recommended_role = "roles/aiplatform.user"
      role_desc = "제미나이 호출 및 Vertex AI 제품군 리소스 사용 권한이다."
    elif service_name == "bigquery.googleapis.com":
      recommended_role = "roles/bigquery.dataEditor"
      role_desc = "빅쿼리 데이터세트 수정 및 테이블 데이터 편집 권한이다."
    elif service_name == "storage.googleapis.com":
      recommended_role = "roles/storage.objectViewer"
      role_desc = "클라우드 스토리지 버킷 객체 조회 뷰어 권한이다."
    elif service_name == "logging.googleapis.com":
      recommended_role = "roles/logging.viewer"
      role_desc = "클라우드 로깅 로그 조회 뷰어 권한이다."
      
    member_type = "user"
    if "gserviceaccount.com" in principal_email:
      member_type = "serviceAccount"
      
    print(f"  [정밀 처방 역할 추천]")
    print(f"    - 권장 역할: {recommended_role}")
    print(f"    - 역할 상세: {role_desc}\n")
    print(f"  [즉시 조치 가능한 원클릭 gcloud 해결 명령어]")
    print(f"    gcloud projects add-iam-policy-binding \"{project_id}\" \\")
    print(f"      --member=\"{member_type}:{principal_email}\" \\")
    print(f"      --role=\"{recommended_role}\"")
    print("-" * 72)
    
  if found_attempts == 0:
    print(f"[성공] 데이터 액세스 권한 거부 감사 로그 실패 사례가 발견되지 않았다. 안전하다!")
    print(f"     (GCP IAM 콘솔 주소: https://console.cloud.google.com/iam-admin/iam )")
    
except Exception as e:
  print(f"[오류] 로그를 조회하는 중 에러가 발생했다. 자격 증명 또는 권한 설정을 점검하기 바란다: {e}")
